# Distilling Models for Efficient Reasoning

In [1]:
import pathlib
import torch
import sympy
import tokenizers

We will be implementing hard distillation
- Hard: argmax of teacher logits only
- Soft: entire softmax distribution of teacher logits

#### Loading dataset

In [2]:
import json
import requests
from pathlib import Path


def load_distill_data(
    local_path=None,
    partition="deepseek-r1-math-train",
    save_copy=True,
):

    if local_path is None:
        local_path = f"{partition}.json"
    local_path = Path(local_path)

    url = (
        "https://huggingface.co/datasets/rasbt/math_distill"
        "/resolve/main/data/"
        f"{partition}.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        f"MATH/{partition}.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB (cached)")
        return data

    assert partition in (
        "deepseek-r1-math-train",
        "deepseek-r1-math500",
         "qwen3-235b-a22b-math-train",
        "qwen3-235b-a22b-math500",
    )

    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        print("Using backup URL.")
        r = requests.get(backup_url, timeout=30)
        r.raise_for_status()

    data = r.json()

    if save_copy:
        with local_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

        size_kb = local_path.stat().st_size / 1e3
        print(f"{local_path}: {size_kb:.1f} KB")

    return data

In [3]:
math_train = load_distill_data(partition="deepseek-r1-math-train")
print("Dataset size:", len(math_train))

deepseek-r1-math-train.json: 107610.0 KB (cached)
Dataset size: 12000


In [4]:
from pprint import pprint
pprint(math_train[67])

{'gtruth_answer': '-112',
 'message_content': 'To find the value of \\((p + 3)(q + 3)\\) where \\(p\\) '
                    'and \\(q\\) are the solutions to the equation \\((x - '
                    '5)(2x + 9) = x^2 - 13x + 40\\), we start by expanding and '
                    'simplifying the equation.\n'
                    '\n'
                    'First, expand the left-hand side (LHS):\n'
                    '\\[\n'
                    '(x - 5)(2x + 9) = 2x^2 + 9x - 10x - 45 = 2x^2 - x - 45\n'
                    '\\]\n'
                    '\n'
                    'The right-hand side (RHS) is given as:\n'
                    '\\[\n'
                    'x^2 - 13x + 40\n'
                    '\\]\n'
                    '\n'
                    'Set the equation equal by moving all terms to the '
                    'left-hand side:\n'
                    '\\[\n'
                    '2x^2 - x - 45 - (x^2 - 13x + 40) = 0\n'
                    '\\]\n'
                    'Simp

In [5]:
def format_distilled_answer(text):
    content = text['message_content'].strip()
    if not content:
        raise ValueError("Missing non-empty 'message_content' field.")

    thinking = str(text["message_thinking"]).strip()
    return f"<think>{thinking}</think>\n\n{content}"

In [6]:
print(format_distilled_answer(math_train[67]))

<think>Okay, let's see. I need to find the two distinct solutions p and q of the equation (x-5)(2x+9) = x² -13x +40, and then compute (p +3)(q +3). Hmm. Alright, first step, maybe I should expand the left side of the equation so that I can form a standard quadratic equation. Then I can solve for p and q using the quadratic formula or maybe factor if possible, and then use Vieta's formulas to find p+q and pq to compute the desired expression. Let me try that.

Starting with the left-hand side (LHS): (x -5)(2x +9). Let me multiply those terms out. Using the distributive property (FOIL), multiply x by 2x gives 2x², x times 9 is 9x, then -5 times 2x is -10x, and -5 times 9 is -45. So adding all those terms:

LHS = 2x² +9x -10x -45. Combine like terms: 9x -10x is -x. So LHS simplifies to 2x² -x -45.

The right-hand side (RHS) is given as x² -13x +40. So the equation becomes:

2x² -x -45 = x² -13x +40

Now, let's subtract RHS from both sides to bring all terms to the left:

2x² -x -45 - x² +

In [7]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (load_model_and_tokenizer, Qwen3Tokenizer)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using CPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


In [8]:
tokenizer.encode('<|im_start|>')

[151644]

In [9]:
tokenizer.encode('<think>')

[13708, 766, 29]

In [10]:
tokenizer._tok.add_special_tokens(['<think>', '</think>'])

2

In [11]:
print(tokenizer.decode([151665, 198, 151666]))

<think>
</think>


In [12]:
tokenizer.eos_token

'<|endoftext|>'

In [13]:
from utils import render_prompt

def build_examples(dataset):
    examples = []
    skipped = 0

    for data in dataset:
        try:
            problem = render_prompt(data['problem'])
            problem_enc = tokenizer.encode(problem)
            problem_id = tokenizer.encode('<|im_start|>user') + tokenizer.encode('\n') + problem_enc + tokenizer.encode('<|im_end|>')

            answer = format_distilled_answer(data)
            answer_enc = tokenizer.encode(answer)
            answer_id = tokenizer.encode('<|im_start|>assistant') + tokenizer.encode('\n') + answer_enc + tokenizer.encode('<|im_end|>')

            token_id = problem_id + tokenizer.encode('\n') + answer_id + [tokenizer.eos_token_id]

            if len(token_id) < 2:
                skipped += 1
                continue

            else:
                examples.append({
                "token_ids": token_id,
                "prompt_len": len(problem_id),
            })
        
        except (KeyError, TypeError, ValueError):
            skipped += 1
    
    return examples, skipped

In [14]:
examples, num_skipped = build_examples(math_train)
num_skipped

0

In [15]:
print(tokenizer.decode(examples[0]['token_ids']))

<|im_start|>user
You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).

Answer:<|im_end|>
<|im_start|>assistant
<think>Okay, so I need to find the values of 'a' and 'b' such that the piecewise function f(x) is continuous everywhere. Then add them together for the final answer. Let me recall that for a piecewise function to be continuous, the left-hand limit and right-hand limit at each boundary point must equal the function's value at that point. The boundaries here are at x = -2 and x = 2. Let me start by checking each boundary point.

First, let's look at x = -2. The function is defined as 2x - b when x < -2, and as x - 5 when -2 ≤ x ≤

#### Filter + Splitting Dataset

In [16]:
def compute_length(dataset):
    lengths = []
    for data in dataset:
        total = len(data["token_ids"])
        length = total - data["prompt_len"]
        lengths.append(length)
    
    avg_len = round(sum(lengths) / len(lengths))

    shortest_len = min(lengths)
    longest_len = max(lengths)
    shortest_idx = lengths.index(shortest_len)
    longest_idx = lengths.index(longest_len)

    print(f"Average: {avg_len} tokens")
    print(f"Shortest: {shortest_len} tokens (index {shortest_idx})")
    print(f"Longest: {longest_len} tokens (index {longest_idx})")

In [17]:
compute_length(examples)

Average: 2839 tokens
Shortest: 191 tokens (index 10846)
Longest: 41829 tokens (index 2529)


In [18]:
def filter(examples, max_len = 2048):
    filtered = [x for x in examples if len(x['token_ids']) < max_len]

    print("Original:", len(examples))
    print("Filtered:", len(filtered))
    print("Removed:", len(examples) - len(filtered))

    return filtered


filtered_examples = filter(examples)

Original: 12000
Filtered: 6694
Removed: 5306


In [19]:
compute_length(filtered_examples)

Average: 1098 tokens
Shortest: 191 tokens (index 5970)
Longest: 1987 tokens (index 2611)


In [20]:
import random

rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]


print("Number of train examples:", len(train_examples))
print("Number of validation examples:", len(val_examples))

Number of train examples: 6669
Number of validation examples: 25


In [21]:
print(tokenizer.decode(filtered_examples[5970]['token_ids']))

<|im_start|>user
You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Let $z$ and $w$ be complex numbers such that $|z| = 2$ and $|w| = 5.$  Find the largest possible value of $|z + w|.$

Answer:<|im_end|>
<|im_start|>assistant
<think>Okay, let's see. I need to find the largest possible value of |z + w| where |z| is 2 and |w| is 5. Hmm, complex numbers... Modulus... Oh right, the triangle inequality! Wait, triangle inequality says that |a + b| ≤ |a| + |b|. So in this case, |z + w| ≤ |z| + |w| = 2 + 5 = 7. So the maximum possible value is 7? But wait, is there a case where this equality holds?

Right, equality in the triangle inequality occurs when the vectors are in the same direction, right? So if z and w are positive real numbers aligned in the same direction, their moduli just add up. For complex numbers, if they have the same argument, meaning they point in the same direction in the complex plane, then their ma

### Just learnt that hard distillation is glorified SFT so everything remains the same

In [27]:
def compute_loss(model, example, device):
    token_id = example['token_ids']
    input = torch.tensor(token_id[:-1], dtype=torch.long, device=device).unsqueeze(0)
    targets = torch.tensor(token_id[1:], dtype= torch.long, device=device)
    output_logits = model(input).squeeze(0)
    
    answer_start = max(example['prompt_len'], 0)
    answer_logits = output_logits[answer_start:]
    answer_targets = targets[answer_start:]

    loss = torch.nn.functional.cross_entropy(input= answer_logits, target= answer_targets)
    return loss

In [28]:
with torch.no_grad():
    loss = compute_loss(
        model, train_examples[5730], device
    )

print(f"Loss: {loss:.2f}")

Loss: 1.34


In [30]:
@torch.no_grad()
def evaluate_examples(model, examples, device):
    training_bool = model.training
    model.eval()
    loss = 0
    num_examples = 0
    for example in examples:
        loss += compute_loss(model, example, device)
        num_examples += 1

    if training_bool:
        model.train()

    return loss/num_examples

In [31]:
train_loss = evaluate_examples(model, train_examples[:3], device)
print(f"Train loss (3 examples): {train_loss:.2f}")

Train loss (3 examples): 0.98


### Training Loop

In [ ]:
def train_distillation(model, train_data, val_data, device, clip = None, epochs = 2, lr = 1e-4, log_every = 100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    train_losslist, val_losslist = [], []
    total_steps = epochs*len(train_data)
    step = 0

    for i in range(epochs):
        train_data = list(train_data)
        rng.shuffle(train_data)

        for data in train_data:
            step += 1
            optimizer.zero_grad()
            train_loss = evaluate_examples(model, data, device)
            train_loss.backward()

            #optional clip
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

            optimizer.step()
            train_losslist.append(train_loss.item())


            if step%log_every == 0:
                model.eval()
                with torch.no_grad():
                    val_loss = evaluate_examples(model, val_data, device)
                model.train()
                val_losslist.append(val_loss.item())
                print(
                    f"[Epoch {i+1}/{epochs} "
                    f"Step {step}/{total_steps}] "
                    f"train_loss={train_loss.item():.4f} "
                    f"val_loss={val_loss.item():.4f}")

    return model